# 607. Sales Person

## Problem
SalesPerson(sales_id,name,salary,commission_rate,hire_date), Company(com_id,name,city), Orders(order_id,order_date,com_id,sales_id,amount): return names of salespeople who made no sales to company RED. Include people with no orders.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/0600-0699/0607.Sales%20Person/README_EN.md) · [LeetCode](https://leetcode.com/problems/sales-person/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/sales-person/solutions/)

Community links are for further reading; the implementations below are independently written.

## Practice first
Run the data cell and try the blank cell before reading solutions.

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

sales_person = pd.DataFrame([[1,"A",100,1,"2024-01-01"],[2,"B",100,1,"2024-01-01"],[3,"C",100,1,"2024-01-01"]], columns=["sales_id","name","salary","commission_rate","hire_date"])
company = pd.DataFrame([[1,"RED","X"],[2,"BLUE","Y"]], columns=["com_id","name","city"])
orders = pd.DataFrame([[1,"2024-02-01",1,1,100],[2,"2024-02-01",2,1,200],[3,"2024-02-01",2,2,100]], columns=["order_id","order_date","com_id","sales_id","amount"])
expected = pd.DataFrame([["B"],["C"]], columns=["name"])
display(sales_person)
display(company)
display(orders)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Nested isin
先找 RED 公司，再找不符合條件的業務。

In [ ]:
def solution_1(sales_person, company, orders):
    red = company.loc[company.name.eq('RED'),'com_id']
    bad = orders.loc[orders.com_id.isin(red),'sales_id']
    return sales_person.loc[~sales_person.sales_id.isin(bad),['name']]

result_1 = solution_1(sales_person.copy(), company.copy(), orders.copy())
display(result_1)

### Solution 2 — Merge and anti-join
anti-join 不會漏掉完全沒有訂單的人。

In [ ]:
def solution_2(sales_person, company, orders):
    bad = orders.merge(company.loc[company.name.eq('RED'),['com_id']],on='com_id')[['sales_id']].drop_duplicates()
    m = sales_person.merge(bad,on='sales_id',how='left',indicator=True)
    return m.loc[m['_merge'].eq('left_only'),['name']]

result_2 = solution_2(sales_person.copy(), company.copy(), orders.copy())
display(result_2)

### Solution 3 — Group any and map
每人是否曾經賣給 RED；未下單 map 後不是 True。

In [ ]:
def solution_3(sales_person, company, orders):
    m = orders.merge(company,on='com_id',how='left')
    g = m.assign(is_red=m.name.eq('RED')).groupby('sales_id').is_red.any()
    return sales_person.loc[~sales_person.sales_id.map(g).eq(True),['name']]

result_3 = solution_3(sales_person.copy(), company.copy(), orders.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.